# Milestone 6C: Distributed Machine Learning with PySpark MLlib

This notebook trains and benchmarks distributed classification models for AML money mule detection using **pyspark.ml**.

### Models Evaluated:
- **Spark Gradient-Boosted Trees (GBTClassifier)**
- **Spark Random Forest (RandomForestClassifier)**
- **Spark Regularized Logistic Regression (LogisticRegression)**

### Key Highlights:
- `VectorAssembler` and `StandardScaler` feature preprocessing pipeline.
- Evaluated with **Precision-Recall AUC (PR-AUC)** via `BinaryClassificationEvaluator`.
- Cross-model comparison against the single-node XGBoost baseline and PyG GraphSAGE GNN.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.spark_ml_pipeline import create_spark_session, SparkMLMuleClassifier

spark = create_spark_session('Databricks-M6C-MLlib')

features_path = '../data/processed/account_features.csv'
split_path = '../data/processed/split_accounts.csv'

raw_features = spark.read.option('header', 'true').option('inferSchema', 'true').csv(features_path)
split_df = spark.read.option('header', 'true').csv(split_path)

classifier = SparkMLMuleClassifier(spark)
train_df, val_df, test_df = classifier.prepare_data(raw_features, split_df)


## 1. Distributed Model Training Suite
We train GBT, Random Forest, and Logistic Regression on distributed partitions.


In [ ]:
# Train models on sample for fast demonstration
sample_train = train_df.sample(False, 0.25, seed=42)
sample_val = val_df.sample(False, 0.50, seed=42)
sample_test = test_df.sample(False, 0.50, seed=42)

results = classifier.run_benchmark_suite(sample_train, sample_val, sample_test)
classifier.print_comparison_table(results)


## 2. Model Persistence & Governance
We save the trained Spark ML pipeline to disk for production serving.


In [ ]:
best_gbt = [r for r in results if r.model_name == 'Spark GBT'][0]
classifier.save_model(best_gbt.model_instance, '../outputs/models/spark_gbt_pipeline')
print('Spark MLlib Pipeline saved successfully.')
spark.stop()
